# SesTemiz — Demo

Gürültülü konuşmadan temiz konuşmayı kestiren U-Net modelinin canlı demosu.

Bu notebook şunları yapar:
1. En iyi iki modeli yükler (`e3_depth5`: PESQ/STOI'de en iyi, `e1_si_snr`: SI-SNR'da en iyi).
2. Validation bölmesinden alınan örnekleri (`samples/`, her SNR için bir tane) temizler ve **dinletir**.
3. Temiz / gürültülü / maske / temizlenmiş spektrogramları **önce–sonra** olarak gösterir.
4. Tüm deneylerin sonuç tablosunu (`docs/results.md`) gösterir.
5. Kendi kaydınızı (ör. telefonla kaydedilmiş gerçek ortam sesi) temizlemenize izin verir.

Modeller yerelde (`experiments/`) yoksa GitHub Release'ten `models/` klasörüne indirilir;
temiz bir klonda (ör. Colab) ek bir şey gerekmez.

In [ ]:
# Colab'da: önce repoyu klonlayıp bağımlılıkları kurun
# !git clone https://github.com/fatihkadim/speech-enhancement.git && %cd speech-enhancement
# !pip install -q -r requirements.txt pesq

## 1. Kurulum

In [ ]:
import logging
import os
import sys
from pathlib import Path

# Notebook notebooks/ içinden ya da repo kökünden açılabilir
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import Audio, Markdown, display

from src.evaluate import compute_metrics
from src.infer import enhance
from src.models.enhancer import load_model
from src.utils.audio import load_audio
from src.utils.plotting import plot_spectrogram, spectrogram_db

%matplotlib inline
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("kök:", ROOT, "| cihaz:", DEVICE)

## 2. Modelleri yükle

Önce yerel deney çıktısına (`experiments/<deney>/seed42/best.pt`) bakılır; yoksa Release'ten indirilir.

In [ ]:
RELEASE_URL = "https://github.com/fatihkadim/speech-enhancement/releases/download/modeller-v1"
MODEL_NAMES = ["e3_depth5", "e1_si_snr"]
SEED = 42


def model_path(name: str) -> Path:
    """Yerel checkpoint'i döndürür; yoksa Release'ten models/ altına indirir."""
    local = ROOT / "experiments" / name / f"seed{SEED}" / "best.pt"
    if local.exists():
        return local
    cached = ROOT / "models" / f"{name}_seed{SEED}.pt"
    if not cached.exists():
        cached.parent.mkdir(exist_ok=True)
        torch.hub.download_url_to_file(f"{RELEASE_URL}/{cached.name}", str(cached))
    return cached


models = {}
for name in MODEL_NAMES:
    models[name], cfg = load_model(model_path(name), DEVICE)
    n_params = sum(p.numel() for p in models[name].parameters())
    print(f"{name}: depth={cfg['model']['depth']}, kayıp={cfg['loss']['name']}, {n_params / 1e6:.1f} M parametre")

SR, N_FFT, HOP = cfg["data"]["sr"], cfg["stft"]["n_fft"], cfg["stft"]["hop"]

## 3. Örnekleri temizle

`samples/` altındaki çiftler validation bölmesinden (-5 … 15 dB) üretildi; modeller bu kayıtları eğitimde görmedi.
Metrikler tam kayıt üzerinde hesaplanır (PESQ kurulu değilse `nan`).

In [ ]:
pairs = sorted(ROOT.glob("samples/*_noisy.wav"), key=lambda p: int(p.name.split("_")[0][3:]))
examples = []
for noisy_path in pairs:
    clean_path = noisy_path.with_name(noisy_path.name.replace("_noisy", "_clean"))
    noisy, clean = load_audio(noisy_path, SR), load_audio(clean_path, SR)
    ex = {"ad": noisy_path.stem.removesuffix("_noisy"), "noisy": noisy, "clean": clean,
          "est": {name: enhance(m, noisy) for name, m in models.items()}}
    examples.append(ex)

rows = []
for ex in examples:
    for name, wave in [("gürültülü (E0)", ex["noisy"]), *ex["est"].items()]:
        rows.append({"örnek": ex["ad"], "sistem": name, **compute_metrics(wave, ex["clean"], SR)})
metrics = pd.DataFrame(rows)
table = metrics.pivot(index="örnek", columns="sistem", values=["si_snr", "pesq", "stoi"])
columns = pd.MultiIndex.from_product([["si_snr", "pesq", "stoi"], ["gürültülü (E0)", *models]])
display(table.reindex(index=[ex["ad"] for ex in examples], columns=columns).round(2))

## 4. Dinle: temiz / gürültülü / temizlenmiş

In [ ]:
for ex in examples:
    display(Markdown(f"### {ex['ad']}"))
    for label, wave in [("temiz (hedef)", ex["clean"]), ("gürültülü (girdi)", ex["noisy"]),
                        *[(f"temizlenmiş — {n}", w) for n, w in ex["est"].items()]]:
        display(Markdown(f"**{label}**"), Audio(wave.numpy(), rate=SR))

## 5. Spektrogram: önce / sonra

Tüm paneller aynı dB ölçeğinde. Maske, modelin her zaman-frekans hücresini ne kadar koruduğunu gösterir
(sarı = koru, mor = sil); `est_mag / |X|` olarak hesaplanır.

In [ ]:
SHOW_MODEL = "e3_depth5"
EPS = 1e-8


def show_before_after(clean: torch.Tensor | None, noisy: torch.Tensor, est: torch.Tensor, model, title: str) -> None:
    """(Temiz) / gürültülü / maske / temizlenmiş spektrogramlarını alt alta çizer."""
    with torch.no_grad():
        out = model(noisy[None].to(DEVICE))
    mag = model.magnitude(noisy[None].to(DEVICE))
    mask = (out["est_mag"] / (mag + EPS)).clamp(0, 1)[0].cpu()
    waves = [w for w in (clean, noisy, est) if w is not None]
    vmax = float(max(spectrogram_db(w, N_FFT, HOP).max() for w in waves))
    panels = ([("temiz (hedef)", clean)] if clean is not None else []) + [("gürültülü (girdi)", noisy), ("maske", None),
                                                                           ("temizlenmiş (çıktı)", est)]
    fig, axes = plt.subplots(len(panels), 1, figsize=(10, 2.3 * len(panels)), sharex=True, constrained_layout=True)
    for ax, (label, wave) in zip(axes, panels):
        if wave is None:
            im = ax.imshow(mask.numpy(), origin="lower", aspect="auto", cmap="viridis", vmin=0, vmax=1,
                           extent=(0, noisy.numel() / SR, 0, SR / 2 / 1000))
            ax.set(title="modelin maskesi (sarı = koru, mor = sil)", ylabel="frekans (kHz)")
            fig.colorbar(im, ax=ax, fraction=0.02)
        else:
            plot_spectrogram(ax, wave, SR, N_FFT, HOP, label, vmax)
    fig.suptitle(title)
    plt.show()


for ex in examples:
    show_before_after(ex["clean"], ex["noisy"], ex["est"][SHOW_MODEL], models[SHOW_MODEL],
                      f"{ex['ad']} — {SHOW_MODEL}")

## 6. Tüm deneylerin sonuçları

`src/collect_results.py`'nin ürettiği tablo (3 tohum ortalaması ± std, test kayıtlarının tamamı).
Deneyleri yeniden çalıştırdıysanız tabloyu güncellemek için:
`uv run python -m src.collect_results --matrix configs/experiments.yaml --markdown docs/results.md`

In [ ]:
display(Markdown((ROOT / "docs" / "results.md").read_text(encoding="utf-8")))

## 7. Kendi kaydını temizle (E7: gerçek kayıt)

`data/raw/gercek_kayit/` altına koyduğunuz `.wav` dosyaları (veya `MY_FILES` listesine eklediğiniz yollar)
temizlenir. Temiz referans olmadığından metrik hesaplanmaz; sonuç kulakla ve spektrogramla değerlendirilir.
Kayıt modelin örnekleme hızına (16 kHz, mono) dönüştürülür.

In [ ]:
MY_FILES: list[Path] = []  # ör. [Path("kayit.wav")]
files = MY_FILES + sorted((ROOT / "data" / "raw" / "gercek_kayit").glob("*.wav"))
if not files:
    print("Gerçek kayıt yok: data/raw/gercek_kayit/ altına .wav koyun veya MY_FILES'a ekleyin.")
for path in files:
    noisy = load_audio(path, SR)
    est = enhance(models[SHOW_MODEL], noisy)
    est = est / max(1.0, float(est.abs().max()))  # kliplenmeyi önle
    display(Markdown(f"### {path.name}"), Markdown("**gürültülü**"), Audio(noisy.numpy(), rate=SR),
            Markdown(f"**temizlenmiş — {SHOW_MODEL}**"), Audio(est.numpy(), rate=SR))
    show_before_after(None, noisy, est, models[SHOW_MODEL], f"{path.name} — {SHOW_MODEL}")